# Accepted Loan Model Tuning

Tune baseline classifiers for `target_bad` prediction with a focus on validation F1 and precision.

Models tuned:

- Logistic Regression
- HistGradientBoostingClassifier
- Random Forest

The notebook uses the preprocessed baseline matrices and keeps validation/test evaluation on full chronological splits.

## 1. Configuration

Set focused hyperparameter grids and runtime controls. Training samples are stratified from the chronological training period only; validation and test remain full-size.

In [1]:
from __future__ import annotations

import json
import os
import time
from itertools import product
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/lendingclub_mplconfig")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 160)
pd.set_option("display.width", 220)

RANDOM_STATE = 42
SEARCH_SAMPLE_ROWS = 200_000
TARGET_PRECISION_FOR_OPERATING_POINT = 0.40

DEFAULT_PROJECT_ROOT = Path(
    "/Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/"
    "Final_Project/Final/CreditRiskRAG"
)

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "Modeling").exists() and (candidate / "README.md").exists():
            return candidate
    return DEFAULT_PROJECT_ROOT

PROJECT_ROOT = find_project_root()
PREPROCESSING_DATASET_DIR = PROJECT_ROOT / "Modeling" / "Preprocessing" / "preprocessing_outputs" / "datasets"
TUNING_OUTPUT_ROOT = PROJECT_ROOT / "Modeling" / "modeling_outputs" / "tuning"
TABLE_DIR = TUNING_OUTPUT_ROOT / "tables"
PLOT_DIR = TUNING_OUTPUT_ROOT / "plots"
MODEL_DIR = TUNING_OUTPUT_ROOT / "models"
for path in [TABLE_DIR, PLOT_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Preprocessing datasets:", PREPROCESSING_DATASET_DIR)
print("Tuning outputs:", TUNING_OUTPUT_ROOT)

PROJECT_ROOT: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG
Preprocessing datasets: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/Preprocessing/preprocessing_outputs/datasets
Tuning outputs: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning


## 2. Load Baseline Preprocessed Data

Use the baseline matrices created by preprocessing. The high-missingness challenger is intentionally not tuned here.

In [2]:
def save_table(df: pd.DataFrame, name: str, index: bool = False) -> Path:
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=index)
    print("Saved:", path)
    return path

def load_parquet(name: str) -> pd.DataFrame:
    path = PREPROCESSING_DATASET_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing preprocessing dataset: {path}")
    return pd.read_parquet(path)

X_train = load_parquet("baseline_train_X.parquet")
X_valid = load_parquet("baseline_validation_X.parquet")
X_test = load_parquet("baseline_test_X.parquet")
y_train = load_parquet("train_y.parquet")["target_bad"].astype(int)
y_valid = load_parquet("validation_y.parquet")["target_bad"].astype(int)
y_test = load_parquet("test_y.parquet")["target_bad"].astype(int)

for split_name, X_part, y_part in [("train", X_train, y_train), ("validation", X_valid, y_valid), ("test", X_test, y_test)]:
    if len(X_part) != len(y_part):
        raise ValueError(f"{split_name} row mismatch: X={len(X_part)}, y={len(y_part)}")
    missing = int(X_part.isna().sum().sum())
    if missing:
        raise ValueError(f"{split_name} has {missing} missing values after preprocessing")

input_summary = pd.DataFrame([
    {"split": "train", "rows": len(X_train), "columns": X_train.shape[1], "bad_rate": round(float(y_train.mean()), 6)},
    {"split": "validation", "rows": len(X_valid), "columns": X_valid.shape[1], "bad_rate": round(float(y_valid.mean()), 6)},
    {"split": "test", "rows": len(X_test), "columns": X_test.shape[1], "bad_rate": round(float(y_test.mean()), 6)},
])
save_table(input_summary, "tuning_input_summary")
display(input_summary)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/tables/tuning_input_summary.csv


,split,rows,columns,bad_rate
0,train,962641,100,0.188300
1,validation,186920,100,0.246763
2,test,195749,100,0.210315


## 3. Tuning Utilities

Search uses validation F1 as the primary score and tracks precision. Thresholds are selected on validation, never test.

In [3]:
def compute_sample_weight(y: pd.Series) -> np.ndarray:
    counts = y.value_counts().to_dict()
    n = len(y)
    n_classes = len(counts)
    return y.map({cls: n / (n_classes * count) for cls, count in counts.items()}).astype("float32").to_numpy()


def stratified_train_sample(X: pd.DataFrame, y: pd.Series, max_rows: int | None, random_state: int) -> tuple[pd.DataFrame, pd.Series]:
    if max_rows is None or len(X) <= max_rows:
        return X, y
    frac = max_rows / len(X)
    idx = y.to_frame("target_bad").groupby("target_bad", group_keys=False).sample(frac=frac, random_state=random_state).index
    return X.loc[idx], y.loc[idx]


def predict_positive_probability(model, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "n_jobs"):
        model.n_jobs = 1
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    scores = model.decision_function(X)
    return 1 / (1 + np.exp(-scores))


def threshold_for_best_f1(y_true: pd.Series, y_score: np.ndarray) -> tuple[float, float, float, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return 0.5, 0.0, 0.0, 0.0
    f1 = (2 * precision[:-1] * recall[:-1]) / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    idx = int(np.nanargmax(f1))
    return float(thresholds[idx]), float(f1[idx]), float(precision[idx]), float(recall[idx])


def threshold_for_target_precision(y_true: pd.Series, y_score: np.ndarray, target_precision: float) -> tuple[float, float, float, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return 0.5, 0.0, 0.0, 0.0
    candidate = np.where(precision[:-1] >= target_precision)[0]
    if len(candidate) == 0:
        return threshold_for_best_f1(y_true, y_score)
    idx = int(candidate[np.argmax(recall[candidate])])
    f1 = (2 * precision[idx] * recall[idx]) / max(precision[idx] + recall[idx], 1e-12)
    return float(thresholds[idx]), float(f1), float(precision[idx]), float(recall[idx])


def score_at_threshold(model_name: str, split: str, y_true: pd.Series, y_score: np.ndarray, threshold: float, operating_point: str) -> dict:
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "split": split,
        "operating_point": operating_point,
        "rows": len(y_true),
        "bad_rate": round(float(y_true.mean()), 6),
        "threshold": round(float(threshold), 6),
        "roc_auc": round(float(roc_auc_score(y_true, y_score)), 6),
        "pr_auc": round(float(average_precision_score(y_true, y_score)), 6),
        "brier_score": round(float(brier_score_loss(y_true, y_score)), 6),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 6),
        "recall": round(float(recall_score(y_true, y_pred, zero_division=0)), 6),
        "f1": round(float(f1_score(y_true, y_pred, zero_division=0)), 6),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def make_model(model_family: str, params: dict):
    if model_family == "logistic_regression":
        return LogisticRegression(
            penalty="l2",
            solver="saga",
            max_iter=400,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            **params,
        )
    if model_family == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            loss="log_loss",
            early_stopping=True,
            validation_fraction=0.1,
            random_state=RANDOM_STATE,
            **params,
        )
    if model_family == "random_forest":
        return RandomForestClassifier(
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            **params,
        )
    raise ValueError(model_family)


def tune_family(model_family: str, param_grid: list[dict], X_fit: pd.DataFrame, y_fit: pd.Series, use_sample_weight: bool) -> tuple[pd.DataFrame, dict]:
    rows = []
    models = {}
    for i, params in enumerate(param_grid, start=1):
        model_name = f"{model_family}_{i:02d}"
        model = make_model(model_family, params)
        fit_kwargs = {}
        if use_sample_weight:
            fit_kwargs["sample_weight"] = compute_sample_weight(y_fit)
        start = time.time()
        model.fit(X_fit, y_fit, **fit_kwargs)
        fit_seconds = round(time.time() - start, 3)
        valid_score = predict_positive_probability(model, X_valid)
        best_threshold, best_f1, best_precision, best_recall = threshold_for_best_f1(y_valid, valid_score)
        precision_threshold, precision_f1, precision_value, precision_recall_value = threshold_for_target_precision(
            y_valid, valid_score, TARGET_PRECISION_FOR_OPERATING_POINT
        )
        rows.append({
            "model_family": model_family,
            "candidate": model_name,
            "params": json.dumps(params, sort_keys=True),
            "fit_rows": len(X_fit),
            "fit_bad_rate": round(float(y_fit.mean()), 6),
            "fit_seconds": fit_seconds,
            "roc_auc": round(float(roc_auc_score(y_valid, valid_score)), 6),
            "pr_auc": round(float(average_precision_score(y_valid, valid_score)), 6),
            "best_f1_threshold": round(best_threshold, 6),
            "best_f1": round(best_f1, 6),
            "best_f1_precision": round(best_precision, 6),
            "best_f1_recall": round(best_recall, 6),
            "target_precision_threshold": round(precision_threshold, 6),
            "target_precision_f1": round(precision_f1, 6),
            "target_precision": round(precision_value, 6),
            "target_precision_recall": round(precision_recall_value, 6),
        })
        models[model_name] = model
        print(f"Finished {model_name}: best_f1={best_f1:.4f}, precision={best_precision:.4f}, recall={best_recall:.4f}, seconds={fit_seconds}")
    results = pd.DataFrame(rows).sort_values(["best_f1", "best_f1_precision", "pr_auc"], ascending=False).reset_index(drop=True)
    return results, models


def evaluate_selected_model(model_name: str, model, best_threshold: float, precision_threshold: float) -> pd.DataFrame:
    rows = []
    for split, X_part, y_part in [("train", X_train, y_train), ("validation", X_valid, y_valid), ("test", X_test, y_test)]:
        score = predict_positive_probability(model, X_part)
        rows.append(score_at_threshold(model_name, split, y_part, score, best_threshold, "best_validation_f1"))
        rows.append(score_at_threshold(model_name, split, y_part, score, precision_threshold, "target_validation_precision"))
    return pd.DataFrame(rows)

## 4. Define Search Grids

Use focused grids instead of exhaustive search. The goal is to improve validation F1/precision without excessive runtime.

In [4]:
logistic_grid = [
    {"C": C, "class_weight": cw}
    for C, cw in product([0.25, 0.5, 1.0, 2.0], [None, "balanced"])
]

hgb_grid = [
    {"learning_rate": lr, "max_iter": it, "max_leaf_nodes": leaves, "l2_regularization": l2}
    for lr, it, leaves, l2 in [
        (0.04, 180, 31, 0.0),
        (0.06, 180, 31, 0.0),
        (0.08, 150, 31, 0.0),
        (0.06, 220, 45, 0.0),
        (0.06, 180, 31, 0.1),
        (0.04, 240, 45, 0.1),
    ]
]

rf_grid = [
    {"n_estimators": n, "max_depth": depth, "min_samples_leaf": leaf, "max_features": mf}
    for n, depth, leaf, mf in [
        (100, 12, 75, "sqrt"),
        (120, 16, 75, "sqrt"),
        (120, 14, 50, 0.35),
    ]
]

grid_summary = pd.DataFrame([
    {"model_family": "logistic_regression", "candidates": len(logistic_grid)},
    {"model_family": "hist_gradient_boosting", "candidates": len(hgb_grid)},
    {"model_family": "random_forest", "candidates": len(rf_grid)},
])
save_table(grid_summary, "tuning_grid_summary")
display(grid_summary)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/tables/tuning_grid_summary.csv


,model_family,candidates
0,logistic_regression,8
1,hist_gradient_boosting,6
2,random_forest,3


## 5. Tune Logistic Regression

Search regularization strength and class weighting. Fit on the full train split because Logistic Regression is relatively cheap.

In [5]:
logistic_results, logistic_models = tune_family(
    "logistic_regression", logistic_grid, X_train, y_train, use_sample_weight=False
)
save_table(logistic_results, "tuning_logistic_regression_candidates")
display(logistic_results)

Finished logistic_regression_01: best_f1=0.4690, precision=0.3499, recall=0.7110, seconds=30.543


Finished logistic_regression_02: best_f1=0.4706, precision=0.3550, recall=0.6978, seconds=27.371


Finished logistic_regression_03: best_f1=0.4690, precision=0.3498, recall=0.7110, seconds=28.328


Finished logistic_regression_04: best_f1=0.4706, precision=0.3550, recall=0.6978, seconds=26.742


Finished logistic_regression_05: best_f1=0.4690, precision=0.3499, recall=0.7110, seconds=26.219


Finished logistic_regression_06: best_f1=0.4706, precision=0.3551, recall=0.6976, seconds=26.382


Finished logistic_regression_07: best_f1=0.4690, precision=0.3499, recall=0.7110, seconds=25.882


Finished logistic_regression_08: best_f1=0.4706, precision=0.3550, recall=0.6978, seconds=27.865
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/tables/tuning_logistic_regression_candidates.csv


,model_family,candidate,params,fit_rows,fit_bad_rate,fit_seconds,roc_auc,pr_auc,best_f1_threshold,best_f1,best_f1_precision,best_f1_recall,target_precision_threshold,target_precision_f1,target_precision,target_precision_recall
0,logistic_regression,logistic_regression_06,"{""C"": 1.0, ""class_weight"": ""balanced""}",962641,0.1883,26.382,0.695409,0.412338,0.471834,0.470642,0.355098,0.697648,0.563545,0.447750,0.400000,0.508444
1,logistic_regression,logistic_regression_08,"{""C"": 2.0, ""class_weight"": ""balanced""}",962641,0.1883,27.865,0.695403,0.412329,0.471747,0.470619,0.355044,0.697756,0.563485,0.447794,0.400003,0.508553
2,logistic_regression,logistic_regression_02,"{""C"": 0.25, ""class_weight"": ""balanced""}",962641,0.1883,27.371,0.695404,0.412331,0.471693,0.470619,0.355021,0.697843,0.563494,0.447783,0.400000,0.508531
3,logistic_regression,logistic_regression_04,"{""C"": 0.5, ""class_weight"": ""balanced""}",962641,0.1883,26.742,0.695403,0.412328,0.471728,0.470616,0.355040,0.697756,0.563492,0.447777,0.400003,0.508509
4,logistic_regression,logistic_regression_07,"{""C"": 2.0, ""class_weight"": null}",962641,0.1883,25.882,0.693012,0.409162,0.168956,0.468977,0.349868,0.711046,0.235486,0.443185,0.400000,0.496824
5,logistic_regression,logistic_regression_05,"{""C"": 1.0, ""class_weight"": null}",962641,0.1883,26.219,0.693009,0.409159,0.168949,0.468967,0.349857,0.711046,0.235270,0.443513,0.400000,0.497648
6,logistic_regression,logistic_regression_01,"{""C"": 0.25, ""class_weight"": null}",962641,0.1883,30.543,0.693011,0.409161,0.168959,0.468966,0.349866,0.711003,0.235309,0.443472,0.400003,0.497539
7,logistic_regression,logistic_regression_03,"{""C"": 0.5, ""class_weight"": null}",962641,0.1883,28.328,0.693010,0.409160,0.168956,0.468953,0.349846,0.711024,0.235258,0.443530,0.400000,0.497691


## 6. Tune HistGradientBoostingClassifier

Search learning rate, iterations, leaf count, and L2 regularization. Fit on the full train split.

In [6]:
hgb_results, hgb_models = tune_family(
    "hist_gradient_boosting", hgb_grid, X_train, y_train, use_sample_weight=True
)
save_table(hgb_results, "tuning_hist_gradient_boosting_candidates")
display(hgb_results)

Finished hist_gradient_boosting_01: best_f1=0.4737, precision=0.3563, recall=0.7064, seconds=23.821


Finished hist_gradient_boosting_02: best_f1=0.4753, precision=0.3578, recall=0.7078, seconds=21.654


Finished hist_gradient_boosting_03: best_f1=0.4754, precision=0.3616, recall=0.6937, seconds=18.818


Finished hist_gradient_boosting_04: best_f1=0.4756, precision=0.3571, recall=0.7117, seconds=26.855


Finished hist_gradient_boosting_05: best_f1=0.4753, precision=0.3571, recall=0.7107, seconds=21.349


Finished hist_gradient_boosting_06: best_f1=0.4755, precision=0.3616, recall=0.6940, seconds=32.258
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/tables/tuning_hist_gradient_boosting_candidates.csv


,model_family,candidate,params,fit_rows,fit_bad_rate,fit_seconds,roc_auc,pr_auc,best_f1_threshold,best_f1,best_f1_precision,best_f1_recall,target_precision_threshold,target_precision_f1,target_precision,target_precision_recall
0,hist_gradient_boosting,hist_gradient_boosting_04,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",962641,0.1883,26.855,0.704363,0.427144,0.467086,0.475623,0.357148,0.711718,0.547510,0.463531,0.400003,0.551046
1,hist_gradient_boosting,hist_gradient_boosting_06,"{""l2_regularization"": 0.1, ""learning_rate"": 0....",962641,0.1883,32.258,0.703919,0.426324,0.479917,0.475452,0.361584,0.694005,0.551052,0.464210,0.400000,0.552976
2,hist_gradient_boosting,hist_gradient_boosting_03,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",962641,0.1883,18.818,0.703726,0.426149,0.479711,0.475363,0.361574,0.693659,0.550881,0.464235,0.400003,0.553041
3,hist_gradient_boosting,hist_gradient_boosting_05,"{""l2_regularization"": 0.1, ""learning_rate"": 0....",962641,0.1883,21.349,0.703222,0.426107,0.472156,0.475334,0.357084,0.710678,0.552125,0.463720,0.400000,0.551588
4,hist_gradient_boosting,hist_gradient_boosting_02,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",962641,0.1883,21.654,0.703167,0.425882,0.473564,0.475318,0.357799,0.707794,0.552058,0.463214,0.400000,0.550157
5,hist_gradient_boosting,hist_gradient_boosting_01,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",962641,0.1883,23.821,0.700946,0.422988,0.473968,0.473675,0.356302,0.706363,0.557939,0.458238,0.400000,0.536325


## 7. Tune Random Forest

Search tree count, depth, leaf size, and feature sampling. Fit on a stratified train-period sample for runtime; evaluate candidates on full validation.

In [7]:
X_rf_search, y_rf_search = stratified_train_sample(X_train, y_train, SEARCH_SAMPLE_ROWS, RANDOM_STATE)
rf_results, rf_models = tune_family(
    "random_forest", rf_grid, X_rf_search, y_rf_search, use_sample_weight=False
)
save_table(rf_results, "tuning_random_forest_candidates")
display(rf_results)

Finished random_forest_01: best_f1=0.4654, precision=0.3531, recall=0.6823, seconds=3.72


Finished random_forest_02: best_f1=0.4678, precision=0.3544, recall=0.6879, seconds=4.956


Finished random_forest_03: best_f1=0.4692, precision=0.3495, recall=0.7136, seconds=13.496
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/tables/tuning_random_forest_candidates.csv


,model_family,candidate,params,fit_rows,fit_bad_rate,fit_seconds,roc_auc,pr_auc,best_f1_threshold,best_f1,best_f1_precision,best_f1_recall,target_precision_threshold,target_precision_f1,target_precision,target_precision_recall
0,random_forest,random_forest_03,"{""max_depth"": 14, ""max_features"": 0.35, ""min_s...",200000,0.1883,13.496,0.694125,0.414834,0.433702,0.469180,0.349467,0.713648,0.527333,0.447531,0.400000,0.507881
1,random_forest,random_forest_02,"{""max_depth"": 16, ""max_features"": ""sqrt"", ""min...",200000,0.1883,4.956,0.693469,0.413912,0.467271,0.467781,0.354374,0.687935,0.539431,0.447413,0.400000,0.507577
2,random_forest,random_forest_01,"{""max_depth"": 12, ""max_features"": ""sqrt"", ""min...",200000,0.1883,3.720,0.691762,0.411960,0.481328,0.465364,0.353086,0.682341,0.549845,0.441811,0.400004,0.493377


## 8. Select And Evaluate Best Candidate Per Family

Pick each family winner by validation F1, then report train/validation/test at two operating points: best validation F1 and target validation precision.

In [8]:
all_candidate_results = pd.concat([logistic_results, hgb_results, rf_results], ignore_index=True)
save_table(all_candidate_results, "tuning_all_candidates")

model_maps = {
    "logistic_regression": logistic_models,
    "hist_gradient_boosting": hgb_models,
    "random_forest": rf_models,
}

family_winners = (
    all_candidate_results.sort_values(["model_family", "best_f1", "best_f1_precision", "pr_auc"], ascending=[True, False, False, False])
    .groupby("model_family", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
save_table(family_winners, "tuning_family_winners")
display(family_winners)

selected_metrics = []
selected_model_paths = []
for row in family_winners.itertuples(index=False):
    model = model_maps[row.model_family][row.candidate]
    eval_df = evaluate_selected_model(row.candidate, model, row.best_f1_threshold, row.target_precision_threshold)
    selected_metrics.append(eval_df)
    model_path = MODEL_DIR / f"{row.candidate}.joblib"
    joblib.dump(model, model_path)
    selected_model_paths.append({"candidate": row.candidate, "model_family": row.model_family, "path": str(model_path)})

selected_metrics = pd.concat(selected_metrics, ignore_index=True)
selected_model_paths = pd.DataFrame(selected_model_paths)
save_table(selected_metrics, "tuning_selected_model_metrics")
save_table(selected_model_paths, "tuning_selected_model_artifacts")
display(selected_metrics)
display(selected_model_paths)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/tables/tuning_all_candidates.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/tables/tuning_family_winners.csv


,model_family,candidate,params,fit_rows,fit_bad_rate,fit_seconds,roc_auc,pr_auc,best_f1_threshold,best_f1,best_f1_precision,best_f1_recall,target_precision_threshold,target_precision_f1,target_precision,target_precision_recall
0,hist_gradient_boosting,hist_gradient_boosting_04,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",962641,0.1883,26.855,0.704363,0.427144,0.467086,0.475623,0.357148,0.711718,0.547510,0.463531,0.400003,0.551046
1,logistic_regression,logistic_regression_06,"{""C"": 1.0, ""class_weight"": ""balanced""}",962641,0.1883,26.382,0.695409,0.412338,0.471834,0.470642,0.355098,0.697648,0.563545,0.447750,0.400000,0.508444
2,random_forest,random_forest_03,"{""max_depth"": 14, ""max_features"": 0.35, ""min_s...",200000,0.1883,13.496,0.694125,0.414834,0.433702,0.469180,0.349467,0.713648,0.527333,0.447531,0.400000,0.507881


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/tables/tuning_selected_model_metrics.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/tables/tuning_selected_model_artifacts.csv


,model,split,operating_point,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp
0,hist_gradient_boosting_04,train,best_validation_f1,962641,0.188300,0.467086,0.741626,0.403284,0.207221,0.303829,0.742432,0.431197,473016,308360,46688,134577
1,hist_gradient_boosting_04,train,target_validation_precision,962641,0.188300,0.547510,0.741626,0.403284,0.207221,0.349301,0.602940,0.442341,577780,203596,71973,109292
2,hist_gradient_boosting_04,validation,best_validation_f1,186920,0.246763,0.467086,0.704363,0.427144,0.217970,0.357134,0.711675,0.475601,81706,59089,13299,32826
3,hist_gradient_boosting_04,validation,target_validation_precision,186920,0.246763,0.547510,0.704363,0.427144,0.217970,0.400016,0.551046,0.463539,102672,38123,20708,25417
4,hist_gradient_boosting_04,test,best_validation_f1,195749,0.210315,0.467086,0.711970,0.381409,0.212843,0.319206,0.708713,0.440162,92352,62228,11992,29177
5,hist_gradient_boosting_04,test,target_validation_precision,195749,0.210315,0.547510,0.711970,0.381409,0.212843,0.357881,0.554082,0.434876,113652,40928,18358,22811
6,logistic_regression_06,train,best_validation_f1,962641,0.188300,0.471834,0.719952,0.372913,0.214035,0.294519,0.713541,0.416942,471559,309817,51925,129340
7,logistic_regression_06,train,target_validation_precision,962641,0.188300,0.563545,0.719952,0.372913,0.214035,0.345536,0.537732,0.420724,596759,184617,83793,97472
8,logistic_regression_06,validation,best_validation_f1,186920,0.246763,0.471834,0.695409,0.412338,0.224055,0.355091,0.697626,0.470631,82354,58441,13947,32178
9,logistic_regression_06,validation,target_validation_precision,186920,0.246763,0.563545,0.695409,0.412338,0.224055,0.400000,0.508444,0.447750,105617,35178,22673,23452


,candidate,model_family,path
0,hist_gradient_boosting_04,hist_gradient_boosting,/Users/lindaperez/Documents/NEU/2026SummerML/M...
1,logistic_regression_06,logistic_regression,/Users/lindaperez/Documents/NEU/2026SummerML/M...
2,random_forest_03,random_forest,/Users/lindaperez/Documents/NEU/2026SummerML/M...


## 9. Compare Against Baseline Notebook Results

Compare validation/test F1 and precision with the previous baseline run if available.

In [9]:
baseline_metrics_path = PROJECT_ROOT / "Modeling" / "modeling_outputs" / "tables" / "baseline_model_metrics.csv"
if baseline_metrics_path.exists():
    baseline_metrics = pd.read_csv(baseline_metrics_path)
    baseline_comp = baseline_metrics[baseline_metrics["split"].isin(["validation", "test"])].copy()
    baseline_comp["source"] = "baseline_notebook"
    tuned_comp = selected_metrics[selected_metrics["split"].isin(["validation", "test"]) & (selected_metrics["operating_point"] == "best_validation_f1")].copy()
    tuned_comp["source"] = "tuned_best_f1"
    comparison_cols = ["source", "model", "split", "precision", "recall", "f1", "roc_auc", "pr_auc", "brier_score", "threshold"]
    comparison = pd.concat([baseline_comp[comparison_cols], tuned_comp[comparison_cols]], ignore_index=True)
    save_table(comparison, "tuning_vs_baseline_comparison")
    display(comparison.sort_values(["split", "source", "f1"], ascending=[True, True, False]))
else:
    print("Baseline metrics not found; skipping comparison.")

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/tables/tuning_vs_baseline_comparison.csv


,source,model,split,precision,recall,f1,roc_auc,pr_auc,brier_score,threshold
3,baseline_notebook,hist_gradient_boosting,test,0.318053,0.707668,0.438864,0.710063,0.378549,0.216326,0.475502
5,baseline_notebook,random_forest,test,0.313066,0.701134,0.432856,0.702540,0.369670,0.208234,0.464443
1,baseline_notebook,logistic_regression,test,0.310553,0.709296,0.431974,0.700063,0.361512,0.225022,0.471834
7,tuned_best_f1,hist_gradient_boosting_04,test,0.319206,0.708713,0.440162,0.711970,0.381409,0.212843,0.467086
11,tuned_best_f1,random_forest_03,test,0.311255,0.711020,0.432973,0.702726,0.369274,0.202167,0.433702
9,tuned_best_f1,logistic_regression_06,test,0.310546,0.709272,0.431962,0.700063,0.361512,0.225022,0.471834
2,baseline_notebook,hist_gradient_boosting,validation,0.357525,0.706168,0.474710,0.702247,0.424887,0.220547,0.475502
0,baseline_notebook,logistic_regression,validation,0.355098,0.697648,0.470642,0.695409,0.412338,0.224055,0.471834
4,baseline_notebook,random_forest,validation,0.351448,0.699512,0.467843,0.693471,0.414000,0.214906,0.464443
6,tuned_best_f1,hist_gradient_boosting_04,validation,0.357134,0.711675,0.475601,0.704363,0.427144,0.217970,0.467086


## 10. Plots And Recommendation

Visualize selected tuned models and recommend the best validation F1 model, with precision tradeoffs documented.

In [10]:
plot_df = selected_metrics[selected_metrics["split"].isin(["validation", "test"])].copy()
for metric in ["f1", "precision", "recall", "pr_auc", "roc_auc"]:
    fig, ax = plt.subplots(figsize=(11, 5))
    pivot = plot_df.pivot_table(index="model", columns=["split", "operating_point"], values=metric, aggfunc="first")
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(f"Tuned Model {metric.upper()} By Split And Operating Point")
    ax.set_ylabel(metric)
    ax.set_xlabel("Model")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="Split / operating point", fontsize=8)
    fig.tight_layout()
    path = PLOT_DIR / f"tuned_model_{metric}_comparison.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)

validation_best = selected_metrics[
    (selected_metrics["split"] == "validation") &
    (selected_metrics["operating_point"] == "best_validation_f1")
].sort_values(["f1", "precision", "pr_auc"], ascending=False).iloc[0]

recommendation = pd.DataFrame([
    {
        "recommended_candidate_by_validation_f1": validation_best["model"],
        "validation_f1": validation_best["f1"],
        "validation_precision": validation_best["precision"],
        "validation_recall": validation_best["recall"],
        "threshold": validation_best["threshold"],
        "selection_basis": "Highest validation F1 among tuned family winners; review target-precision operating point before final threshold freeze.",
    }
])
save_table(recommendation, "tuning_recommendation")
display(recommendation)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/plots/tuned_model_f1_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/plots/tuned_model_precision_comparison.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/plots/tuned_model_recall_comparison.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/plots/tuned_model_pr_auc_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/plots/tuned_model_roc_auc_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tuning/tables/tuning_recommendation.csv


,recommended_candidate_by_validation_f1,validation_f1,validation_precision,validation_recall,threshold,selection_basis
0,hist_gradient_boosting_04,0.475601,0.357134,0.711675,0.467086,Highest validation F1 among tuned family winne...


## 11. Precision At Fixed Review Volumes

Evaluate operating points by review capacity rather than by a single F1 or recall target. For each selected tuned model, sort applications by predicted risk and measure precision/recall if the business reviews only the top 1%, 2%, 5%, 10%, 15%, 20%, 25%, or 30% of validation/test applications.

In [ ]:
REVIEW_RATES = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]


def review_volume_metrics(model_name: str, split: str, y_true: pd.Series, y_score: np.ndarray, review_rates: list[float]) -> pd.DataFrame:
    frame = pd.DataFrame({"target_bad": y_true.to_numpy(), "score": y_score})
    frame = frame.sort_values("score", ascending=False).reset_index(drop=True)
    total_rows = len(frame)
    total_bad = int(frame["target_bad"].sum())
    base_bad_rate = float(frame["target_bad"].mean())
    rows = []
    for rate in review_rates:
        review_count = max(1, int(np.ceil(total_rows * rate)))
        reviewed = frame.iloc[:review_count]
        captured_bad = int(reviewed["target_bad"].sum())
        precision = captured_bad / review_count
        recall = captured_bad / total_bad if total_bad else 0.0
        rows.append({
            "model": model_name,
            "split": split,
            "review_rate": rate,
            "review_pct": round(rate * 100, 2),
            "review_count": review_count,
            "score_threshold_min": round(float(reviewed["score"].min()), 6),
            "base_bad_rate": round(base_bad_rate, 6),
            "precision": round(float(precision), 6),
            "recall": round(float(recall), 6),
            "captured_bad": captured_bad,
            "total_bad": total_bad,
            "lift_over_base_bad_rate": round(float(precision / base_bad_rate), 6) if base_bad_rate else np.nan,
        })
    return pd.DataFrame(rows)


review_volume_tables = []
for row in family_winners.itertuples(index=False):
    model = model_maps[row.model_family][row.candidate]
    for split, X_part, y_part in [("validation", X_valid, y_valid), ("test", X_test, y_test)]:
        scores = predict_positive_probability(model, X_part)
        review_volume_tables.append(review_volume_metrics(row.candidate, split, y_part, scores, REVIEW_RATES))

review_volume_precision = pd.concat(review_volume_tables, ignore_index=True)
save_table(review_volume_precision, "tuning_review_volume_precision")
display(review_volume_precision)

for split in ["validation", "test"]:
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_df = review_volume_precision[review_volume_precision["split"] == split]
    for model_name, group in plot_df.groupby("model"):
        ax.plot(group["review_pct"], group["precision"], marker="o", label=model_name)
    ax.set_title(f"Precision At Fixed Review Volumes ({split})")
    ax.set_xlabel("Applications reviewed (%)")
    ax.set_ylabel("Precision / bad rate among reviewed loans")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    path = PLOT_DIR / f"tuned_model_precision_by_review_volume_{split}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)

recommended_review_volume = review_volume_precision[
    (review_volume_precision["model"] == recommendation.iloc[0]["recommended_candidate_by_validation_f1"]) &
    (review_volume_precision["split"].isin(["validation", "test"]))
].copy()
save_table(recommended_review_volume, "tuning_recommended_model_review_volume_precision")
display(recommended_review_volume)